In [ ]:
import torch
import torch.nn.functional as F

def compute_gae_and_returns(rewards, values, mask, gamma=1.0, lam=0.95):
    """
    计算广义优势估计 (GAE) 和目标价值 (Returns)
    参数:
        rewards: [batch_size, seq_len] 包含KL惩罚和最后一步RM得分的即时奖励
        values:  [batch_size, seq_len] Critic老权重输出的价值预估 V_old(s_t)
        mask:    [batch_size, seq_len] 用于忽略 Padding 的掩码 (1为有效词，0为Padding)
        gamma:   折扣因子 (LLM通常为1.0)
        lam:     GAE的平滑因子 (通常为0.95)
    返回:
        advantages: [batch_size, seq_len] 计算出的 A_t
        returns:    [batch_size, seq_len] 计算出的 V_target,t
    """
    batch_size, seq_len = rewards.shape
    advantages = torch.zeros_like(rewards)
    
    last_gae_lam = 0 # 记录 t+1 步的优势传递
    
    
    # 从最后一个 Token 往前倒序遍历计算
    for t in reversed(range(seq_len)):
        # 如果当前步是序列的最后一步，没有下一个 V 值；否则取 t+1 的 V 值
        next_values = values[:, t + 1] if t < seq_len - 1 else 0.0
        
        # 1. 计算单步 TD Error (时序差分误差)
        # delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)
        delta_t = rewards[:, t] + gamma * next_values - values[:, t]
        
        # 2. 计算 GAE 并累加未来的优势
        # mask[:, t] 用来截断 Padding 部分的优势回传
        last_gae_lam = delta_t + gamma * lam * last_gae_lam * mask[:, t]
        advantages[:, t] = last_gae_lam
        
    # 3. 计算目标价值: V_target = Advantage + V_old
    returns = advantages + values
    
    return advantages, returns

In [ ]:
def compute_ppo_loss(
    log_probs, old_log_probs, values, returns, advantages, mask, 
    clip_range=0.2, value_clip_range=0.2
):
    """
    计算 Actor 的策略损失和 Critic 的价值损失
    参数:
        log_probs:     当前 Actor 网络新生成的对数概率 \pi_\theta
        old_log_probs: 之前收集的旧对数概率 \pi_old
        values:        当前 Critic 网络新生成的价值评估 V_\omega
        returns:       GAE算出的真实回报目标 V_target
        advantages:    GAE算出的优势 \hat{A}_t
    """
    # ====== 1. Actor 策略损失 (Policy Loss) ======
    # 计算新旧策略概率的比值: ratio = exp(log \pi_new - log \pi_old)
    ratio = torch.exp(log_probs - old_log_probs)
    
    # 未截断的 Surrogate Loss
    surr1 = ratio * advantages
    
    # 截断的 Surrogate Loss (防止步子迈太大)
    surr2 = torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range) * advantages
    
    # 取最小值的负数 (因为 PyTorch 是默认做梯度下降最小化，所以加负号变成最大化收益)
    actor_loss = -torch.min(surr1, surr2)
    
    # ====== 2. Critic 价值损失 (Value Loss) ======
    # 直接计算均方误差 (MSE)
    value_loss = 0.5 * (values - returns) ** 2
    
    # 面试加分项: 工业界通常也会对 Value Loss 进行 Clip 操作，防止 Critic 更新过于剧烈
    # 这里为了代码简洁，仅保留最核心的 MSE 逻辑
    
    # ====== 3. 掩码与求平均 ======
    # 只计算有效 Token 的 Loss，忽略 Padding
    actor_loss = (actor_loss * mask).sum() / mask.sum()
    value_loss = (value_loss * mask).sum() / mask.sum()
    
    return actor_loss, value_loss

In [ ]:
def ppo_training_step(actor, critic, ref_model, reward_model, prompt, kl_coef=0.1):
    """
    单次 PPO 训练的完整流水线伪代码
    """
    # ---------------------------------------------------------
    # 阶段 1：无梯度前向采样 (Trajectory Collection)
    # ---------------------------------------------------------
    with torch.no_grad():
        # 1. Actor 生成回答序列
        # answer_seq: [batch_size, seq_len]
        answer_seq = actor.generate(prompt)
        
        # 2. 获取旧策略的对数概率和 Critic 的旧价值预估
        old_log_probs = actor(prompt, answer_seq).log_probs
        old_values = critic(prompt, answer_seq).values
        
        # 3. 获取 Reference 模型的参考概率
        ref_log_probs = ref_model(prompt, answer_seq).log_probs
        
        # 4. 获取 Reward Model 给出的最终大分 (只在序列末尾有值)
        # rm_scores: [batch_size]
        rm_scores = reward_model(prompt, answer_seq)
        
    # ---------------------------------------------------------
    # 阶段 2：计算密集奖励与 GAE
    # ---------------------------------------------------------
    # 构建即时奖励矩阵 (仅 KL 惩罚)
    rewards = -kl_coef * (old_log_probs - ref_log_probs)
    
    # 将 RM 的大分加到序列的最后一个有效 Token 上
    # (假设 get_last_token_index 是一个获取末尾索引的辅助函数)
    last_idx = get_last_token_index(answer_seq) 
    rewards[range(batch_size), last_idx] += rm_scores
    
    # 计算 GAE 和 Returns (这里使用全 1 的掩码作为示例)
    mask = (answer_seq != PAD_TOKEN_ID).float()
    advantages, returns = compute_gae_and_returns(rewards, old_values, mask)
    
    # 对 Advantage 进行白化 (Normalization)，极其关键的工程 Trick，防止梯度爆炸
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    # ---------------------------------------------------------
    # 阶段 3：计算 Loss 并反向传播更新 (梯度下降)
    # ---------------------------------------------------------
    # 开启梯度，前向传播当前最新的 Actor 和 Critic 网络
    # (在真实的 PPO 中，这里通常会循环几个 Epoch / Mini-batch 来复用经验数据)
    current_log_probs = actor(prompt, answer_seq).log_probs
    current_values = critic(prompt, answer_seq).values
    
    # 调用刚才写的 Loss 计算函数
    actor_loss, value_loss = compute_ppo_loss(
        current_log_probs, old_log_probs, current_values, returns, advantages, mask
    )
    
    # 总 Loss
    total_loss = actor_loss + 0.1 * value_loss
    
    # 反向传播与优化器更新
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    return total_loss.item()